# 33 — Integrated Radius Comparison

Scores each radius by combining satellite opportunity and fusion evidence.

In [ ]:

FRAMEWORK_CSV = "outputs/newhaven_radius/radius_framework.csv"
SAT_DAILY_CSV = "outputs/newhaven_multi_radius/satellite_daily_coverage.csv"
SAT_SUMMARY_CSV = "outputs/newhaven_multi_radius/satellite_radius_summary.csv"
FUSION_CSV = "outputs/newhaven_fusion/fusion_daily.csv"
OUTPUT_DIR = "outputs/newhaven_integrated"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

framework,framework_meta=safe_read_csv(FRAMEWORK_CSV)
sat_daily,sat_daily_meta=safe_read_csv(SAT_DAILY_CSV)
sat_summary,sat_summary_meta=safe_read_csv(SAT_SUMMARY_CSV)
fusion,fusion_meta=safe_read_csv(FUSION_CSV)
if sat_daily.empty:
    integrated_daily=pd.DataFrame(columns=["radius_miles","date","scenes","fusion_score","integrated_score"])
else:
    if not fusion.empty and "date" in fusion.columns:
        sat_daily["date"]=pd.to_datetime(sat_daily["date"],errors="coerce").dt.date.astype("string")
        fusion["date"]=pd.to_datetime(fusion["date"],errors="coerce").dt.date.astype("string")
        integrated_daily=sat_daily.merge(fusion[["date","fusion_score","article_count","ground_rows","mean_wind_speed_ms","mean_cloud_cover_pct"]],on="date",how="left")
    else:
        integrated_daily=sat_daily.copy(); integrated_daily["fusion_score"]=np.nan
    integrated_daily["integrated_score"]=pd.to_numeric(integrated_daily["scenes"],errors="coerce").fillna(0)*2.0 + pd.to_numeric(integrated_daily["fusion_score"],errors="coerce").fillna(0)
integrated_daily_csv=Path(OUTPUT_DIR)/"integrated_radius_daily.csv"
integrated_daily.to_csv(integrated_daily_csv,index=False)
display(integrated_daily.head(20))


In [ ]:

if not integrated_daily.empty:
    summary=integrated_daily.groupby("radius_miles",dropna=False).agg(days_with_scenes=("date","nunique"),total_scenes=("scenes","sum"),mean_integrated_score=("integrated_score","mean"),max_integrated_score=("integrated_score","max")).reset_index().sort_values("radius_miles")
else:
    summary=pd.DataFrame(columns=["radius_miles","days_with_scenes","total_scenes","mean_integrated_score","max_integrated_score"])
if not summary.empty:
    summary["specificity_penalty"]=summary["radius_miles"]/10.0
    summary["recommended_score"]=summary["mean_integrated_score"]-summary["specificity_penalty"]
    summary=summary.sort_values(["recommended_score","radius_miles"],ascending=[False,True]).reset_index(drop=True)
summary_csv=Path(OUTPUT_DIR)/"integrated_radius_summary.csv"
summary.to_csv(summary_csv,index=False)
display(summary)


In [ ]:

fig,ax=plt.subplots(figsize=(10,5))
if not summary.empty:
    ax.plot(summary["radius_miles"],summary["recommended_score"],marker="o",label="recommended_score")
    ax.plot(summary["radius_miles"],summary["mean_integrated_score"],marker="o",label="mean_integrated_score")
ax.set_title("Integrated radius comparison"); ax.set_xlabel("Radius (miles)"); ax.set_ylabel("Score")
if not summary.empty: ax.legend()
fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/"integrated_radius_plot.png"; fig.savefig(plot_path,dpi=150); plt.show()
manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"inputs":[framework_meta,sat_daily_meta,sat_summary_meta,fusion_meta],"rows_daily":int(len(integrated_daily)),"rows_summary":int(len(summary)),"outputs":{"integrated_daily_csv":str(integrated_daily_csv),"summary_csv":str(summary_csv),"plot_png":str(plot_path)}}
manifest_path=Path(OUTPUT_DIR)/"integrated_radius_manifest.json"; manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Saved:",manifest_path)
